# eBay Deal Finder — Arbitrage Research Tool

Find underpriced eBay listings by analyzing active market price distributions, with optional AI-powered listing evaluation.

Converted from `src/index.js` to Python using direct REST calls.

## Cell 1: Setup & Dependencies

In [1]:
%pip install requests python-dotenv openai --quiet

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import base64
import json
import time
import math
import statistics
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from dotenv import load_dotenv
from openai import AzureOpenAI

/Users/albertjoseph/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Cell 2: Load .env & Configuration

In [3]:
load_dotenv(override=True)

# Validate required env vars
EBAY_APP_ID = os.environ.get("EBAY_APP_ID", "")
EBAY_CERT_ID = os.environ.get("EBAY_CERT_ID", "")
EBAY_SANDBOX = os.environ.get("EBAY_SANDBOX", "false").lower() == "true"
EBAY_MARKETPLACE_ID = os.environ.get("EBAY_MARKETPLACE_ID", "EBAY_US")

if not EBAY_APP_ID or not EBAY_CERT_ID:
    raise ValueError("EBAY_APP_ID and EBAY_CERT_ID must be set in .env")

# Azure OpenAI (optional)
AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "")
LLM_ENABLED = bool(AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY and AZURE_OPENAI_DEPLOYMENT)

# ─── Search Configuration ───
SEARCH_QUERY = '"TI-84 Plus CE" graphing calculator'
SEARCH_EXCLUSIONS = '-charger -case -cable -cover -emulator -software -cord -bag -pouch -dock -screen -protector -"parts only" -"for parts" -stand -holder -skin -sticker -keypad'
CONDITION = "Used"
MIN_Z_SCORE = -1.0
EBAY_FEE_RATE = 0.13
RESULTS_PER_PAGE = 200
MAX_LLM_CALLS = 200
MAX_IMAGES_PER_LISTING = 5
MIN_MARKET_SAMPLE = 20
MIN_STD_DEV = 0.01
FALLBACK_DISCOUNT_RATE = 0.1
API_TIMEOUT_S = 15
LLM_TIMEOUT_S = 45
API_MAX_RETRIES = 2
API_RETRY_BASE_DELAY_S = 0.5
LLM_CONCURRENCY = 5

# ─── API URLs ───
EBAY_API_BASE = "https://api.sandbox.ebay.com" if EBAY_SANDBOX else "https://api.ebay.com"
EBAY_AUTH_URL = f"{EBAY_API_BASE}/identity/v1/oauth2/token"
EBAY_BROWSE_URL = f"{EBAY_API_BASE}/buy/browse/v1"

print(f"Environment:  {'SANDBOX' if EBAY_SANDBOX else 'PRODUCTION'}")
print(f"Marketplace:  {EBAY_MARKETPLACE_ID}")
print(f"LLM enabled:  {LLM_ENABLED}")
print(f"Search query: {SEARCH_QUERY}")

Environment:  PRODUCTION
Marketplace:  EBAY_US
LLM enabled:  True
Search query: "TI-84 Plus CE" graphing calculator


## Cell 3: eBay OAuth Token

In [4]:
def get_ebay_token():
    """Mint an application access token via client credentials grant."""
    credentials = base64.b64encode(f"{EBAY_APP_ID}:{EBAY_CERT_ID}".encode()).decode()
    resp = requests.post(
        EBAY_AUTH_URL,
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Authorization": f"Basic {credentials}",
        },
        data={
            "grant_type": "client_credentials",
            "scope": "https://api.ebay.com/oauth/api_scope",
        },
        timeout=API_TIMEOUT_S,
    )
    resp.raise_for_status()
    token_data = resp.json()
    return token_data["access_token"], token_data["expires_in"]

ACCESS_TOKEN, expires_in = get_ebay_token()
print(f"✅ Token acquired — expires in {expires_in}s ({expires_in // 3600}h {(expires_in % 3600) // 60}m)")

# Common headers for all Browse API calls
EBAY_HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "X-EBAY-C-MARKETPLACE-ID": EBAY_MARKETPLACE_ID,
    "Content-Type": "application/json",
}

✅ Token acquired — expires in 7200s (2h 0m)


## Cell 4: Helper Functions

In [5]:
def format_currency(n):
    return f"${n:.2f}"


def filter_outliers(prices):
    """Remove outliers using IQR method."""
    sorted_p = sorted(prices)
    n = len(sorted_p)
    q1 = sorted_p[int(n * 0.25)]
    q3 = sorted_p[int(n * 0.75)]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    filtered = [p for p in sorted_p if lower <= p <= upper]
    return {
        "filtered": filtered,
        "lower": max(lower, 0),
        "upper": upper,
        "removed_count": len(prices) - len(filtered),
    }


def std_dev(numbers, mean):
    """Population standard deviation."""
    squared_diffs = [(n - mean) ** 2 for n in numbers]
    return math.sqrt(sum(squared_diffs) / len(numbers))


def call_api_with_retry(label, request_func, timeout_s=API_TIMEOUT_S, max_retries=API_MAX_RETRIES):
    """Call an API function with timeout and exponential backoff retry."""
    for attempt in range(1, max_retries + 2):
        try:
            return request_func()
        except requests.exceptions.RequestException as e:
            status = getattr(e.response, "status_code", None) if hasattr(e, "response") else None
            retryable = status in (408, 429) or (status is not None and status >= 500)
            if not retryable:
                msg = str(e).lower()
                retryable = any(kw in msg for kw in ("timeout", "timed out", "network", "connectionreset"))

            if attempt <= max_retries and retryable:
                wait = API_RETRY_BASE_DELAY_S * (2 ** (attempt - 1))
                print(f"⚠ {label} failed (attempt {attempt}/{max_retries + 1}): {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


def browse_search(params):
    """Call Browse API search endpoint."""
    resp = requests.get(
        f"{EBAY_BROWSE_URL}/item_summary/search",
        headers=EBAY_HEADERS,
        params=params,
        timeout=API_TIMEOUT_S,
    )
    resp.raise_for_status()
    return resp.json()


def browse_get_item(item_id):
    """Call Browse API getItem endpoint."""
    resp = requests.get(
        f"{EBAY_BROWSE_URL}/item/{item_id}",
        headers=EBAY_HEADERS,
        timeout=API_TIMEOUT_S,
    )
    resp.raise_for_status()
    return resp.json()


def normalize_condition(condition):
    return condition.strip().upper().replace(" ", "_")


print("✅ Helper functions defined")

✅ Helper functions defined


## Cell 5: Test API Call — Browse Search

In [6]:
# Quick test: fetch 5 items to validate token & API access
test_results = browse_search({
    "q": SEARCH_QUERY,
    "limit": "5",
    "filter": f"buyingOptions:{{FIXED_PRICE}},conditions:{{{normalize_condition(CONDITION)}}},priceCurrency:USD",
})

items = test_results.get("itemSummaries", [])
total = test_results.get("total", 0)
print(f"✅ Browse API working — {total} total results for \"{SEARCH_QUERY}\"\n")
for item in items:
    price = item.get("price", {})
    print(f"  ${price.get('value', '?')} — {item.get('title', 'N/A')[:70]}")

✅ Browse API working — 683 total results for ""TI-84 Plus CE" graphing calculator"

  $59.99 — Texas Instruments TI-84 Plus CE Graphing Calculator WORKING
  $64.99 — Texas Instruments TI-84 Plus CE Graphing Calculator WORKING w/ Charger
  $35.00 — Texas Instruments TI-84 Plus CE Color Graphing Calculator - Black
  $79.99 — Texas Instruments TI-84 Plus CE Python Color Graphing Calculator MINT
  $50.00 — Texas Instruments Ti 84 Plus Ce Graphing Calculator - Black


## Cell 6: Step 1 — Get Market Listings

In [7]:
def get_market_listings(query, condition):
    """Fetch all active market listings for price distribution analysis."""
    full_query = f"{query} {SEARCH_EXCLUSIONS}" if SEARCH_EXCLUSIONS else query
    condition_filter = normalize_condition(condition)
    print(f"\n📊 Fetching active market listings for \"{query}\" ({condition})...\n")

    all_items = []
    max_items = 10_000
    page = 0

    while len(all_items) < max_items:
        try:
            result = call_api_with_retry(
                f"Browse market listings page {page + 1}",
                lambda p=page: browse_search({
                    "q": full_query,
                    "filter": f"buyingOptions:{{FIXED_PRICE}},conditions:{{{condition_filter}}},priceCurrency:USD",
                    "limit": str(RESULTS_PER_PAGE),
                    "offset": str(p * RESULTS_PER_PAGE),
                }),
            )

            items = result.get("itemSummaries", [])
            if not items:
                break

            all_items.extend(items)
            total = result.get("total", 0)
            if len(all_items) >= total:
                break
            page += 1

        except Exception as e:
            if not all_items:
                raise RuntimeError(f"Unable to fetch market listings page {page + 1}: {e}")
            print(f"⚠ Market listings fetch stopped at page {page + 1}: {e}. Continuing with {len(all_items)} result(s).")
            break

    print(f"📦 Fetched {len(all_items)} market listings across {page + 1} page(s)")
    return all_items


market_items = get_market_listings(SEARCH_QUERY, CONDITION)


📊 Fetching active market listings for ""TI-84 Plus CE" graphing calculator" (Used)...

📦 Fetched 425 market listings across 3 page(s)


## Cell 7: Step 2 — Analyze Market Prices

In [8]:
def analyze_prices(market_items):
    """Analyze market prices: IQR filtering, stats, price bands, market verdict."""
    prices = []
    for item in market_items:
        try:
            p = float(item.get("price", {}).get("value", 0))
            if p > 0:
                prices.append(p)
        except (TypeError, ValueError):
            continue

    if not prices:
        print("❌ No market price data found.")
        return None
    if len(prices) < MIN_MARKET_SAMPLE:
        print(f"❌ Not enough market listings ({len(prices)} found, need {MIN_MARKET_SAMPLE}).")
        return None

    sorted_prices = sorted(prices)
    avg = statistics.mean(prices)
    med = statistics.median(prices)

    print("─" * 50)
    print("  RAW MARKET PRICES (active listings)")
    print("─" * 50)
    print(f"  Items analyzed:  {len(prices)}")
    print(f"  Average price:   {format_currency(avg)}")
    print(f"  Median price:    {format_currency(med)}")
    print(f"  Low / High:      {format_currency(sorted_prices[0])} — {format_currency(sorted_prices[-1])}")
    print("─" * 50)

    result = filter_outliers(prices)
    filtered = result["filtered"]

    if not filtered:
        print("❌ All prices were filtered as outliers.")
        return None
    if len(filtered) < MIN_MARKET_SAMPLE:
        print(f"❌ Too few listings after outlier filtering ({len(filtered)} kept, need {MIN_MARKET_SAMPLE}).")
        return None

    filtered_sorted = sorted(filtered)
    filtered_mean = statistics.mean(filtered)
    filtered_median = statistics.median(filtered)
    filtered_std_dev_raw = std_dev(filtered, filtered_mean)
    has_usable_std_dev = math.isfinite(filtered_std_dev_raw) and filtered_std_dev_raw >= MIN_STD_DEV
    filtered_std = filtered_std_dev_raw if has_usable_std_dev else 0
    cv = filtered_std / filtered_mean if has_usable_std_dev else 0

    stats = {
        "count": len(filtered),
        "average": filtered_mean,
        "median": filtered_median,
        "std_dev": filtered_std,
        "has_usable_std_dev": has_usable_std_dev,
        "cv": cv,
        "low": filtered_sorted[0],
        "high": filtered_sorted[-1],
        "p25": filtered_sorted[int(len(filtered_sorted) * 0.25)],
        "p75": filtered_sorted[int(len(filtered_sorted) * 0.75)],
    }

    print(f"\n  🧹 IQR FILTER: keeping {format_currency(result['lower'])} — {format_currency(result['upper'])}")
    print(f"     Removed {result['removed_count']} outlier(s)\n")
    print("─" * 50)
    print("  FILTERED MARKET VALUE")
    print("─" * 50)
    print(f"  Items kept:      {stats['count']} of {len(prices)}")
    print(f"  Average price:   {format_currency(stats['average'])}")
    print(f"  Median price:    {format_currency(stats['median'])}")
    print(f"  Std deviation:   {format_currency(stats['std_dev'])}")
    print(f"  Low / High:      {format_currency(stats['low'])} — {format_currency(stats['high'])}")
    print(f"  25th / 75th pct: {format_currency(stats['p25'])} — {format_currency(stats['p75'])}")
    print("─" * 50)

    print("\n  📐 PRICE BANDS")
    print("─" * 50)
    if stats["has_usable_std_dev"]:
        print(f"  -2σ (strong buy): {format_currency(stats['median'] - 2 * stats['std_dev'])}")
        print(f"  -1σ (buy):        {format_currency(stats['median'] - 1 * stats['std_dev'])}")
        print(f"   0  (fair value): {format_currency(stats['median'])}")
        print(f"  +1σ (overpriced): {format_currency(stats['median'] + 1 * stats['std_dev'])}")
        print(f"  +2σ (avoid):      {format_currency(stats['median'] + 2 * stats['std_dev'])}")
    else:
        fallback_price = stats["median"] * (1 - FALLBACK_DISCOUNT_RATE)
        print(f"  ⚠ Std deviation is below {MIN_STD_DEV}; using discount scoring instead of z-score.")
        print(f"  Discount threshold ({FALLBACK_DISCOUNT_RATE * 100:.0f}% below median): {format_currency(fallback_price)}")
    print("─" * 50)

    if not stats["has_usable_std_dev"]:
        verdict = "⚠️  Very low variance — using discount-based scoring"
    elif cv < 0.3:
        verdict = "✅ Tight market — good for arbitrage"
    elif cv < 0.5:
        verdict = "⚠️  Moderate spread — proceed with caution"
    else:
        verdict = "❌ High variance — no reliable fair value"

    cv_display = f"{cv:.3f}" if stats["has_usable_std_dev"] else "N/A"
    print(f"\n  CV: {cv_display}  →  {verdict}")
    print("─" * 50)

    return stats


market_stats = analyze_prices(market_items)
if market_stats is None:
    raise SystemExit("Cannot continue without valid market stats.")

──────────────────────────────────────────────────
  RAW MARKET PRICES (active listings)
──────────────────────────────────────────────────
  Items analyzed:  425
  Average price:   $83.43
  Median price:    $75.00
  Low / High:      $7.00 — $899.00
──────────────────────────────────────────────────

  🧹 IQR FILTER: keeping $27.50 — $127.50
     Removed 29 outlier(s)

──────────────────────────────────────────────────
  FILTERED MARKET VALUE
──────────────────────────────────────────────────
  Items kept:      396 of 425
  Average price:   $77.03
  Median price:    $75.00
  Std deviation:   $18.86
  Low / High:      $28.00 — $126.99
  25th / 75th pct: $65.00 — $89.99
──────────────────────────────────────────────────

  📐 PRICE BANDS
──────────────────────────────────────────────────
  -2σ (strong buy): $37.28
  -1σ (buy):        $56.14
   0  (fair value): $75.00
  +1σ (overpriced): $93.86
  +2σ (avoid):      $112.72
──────────────────────────────────────────────────

  CV: 0.245  →  ✅

## Cell 8: Step 3 — Find Deals

In [9]:
def find_deal_listings(query, condition, market_stats):
    """Search for underpriced active listings below the z-score or discount threshold."""
    has_usable = market_stats["has_usable_std_dev"]
    if has_usable:
        raw_max = market_stats["median"] + MIN_Z_SCORE * market_stats["std_dev"]
    else:
        raw_max = market_stats["median"] * (1 - FALLBACK_DISCOUNT_RATE)
    max_price = max(1, int(raw_max))

    condition_filter = normalize_condition(condition)
    explanation = (
        f"(z ≤ {MIN_Z_SCORE}, i.e. {abs(MIN_Z_SCORE)}σ below {format_currency(market_stats['median'])} median)"
        if has_usable
        else f"({FALLBACK_DISCOUNT_RATE * 100:.0f}% below {format_currency(market_stats['median'])} median fallback)"
    )
    print(f"\n🔎 Searching active listings under {format_currency(max_price)} {explanation}...\n")

    full_query = f"{query} {SEARCH_EXCLUSIONS}" if SEARCH_EXCLUSIONS else query
    all_items = []
    max_items = 10_000
    page = 0

    while len(all_items) < max_items:
        try:
            results = call_api_with_retry(
                f"Browse deal search page {page + 1}",
                lambda p=page: browse_search({
                    "q": full_query,
                    "filter": f"buyingOptions:{{FIXED_PRICE}},conditions:{{{condition_filter}}},price:[..{max_price}],priceCurrency:USD",
                    "sort": "price",
                    "limit": str(RESULTS_PER_PAGE),
                    "offset": str(p * RESULTS_PER_PAGE),
                }),
            )
            items = results.get("itemSummaries", [])
            if not items:
                break
            all_items.extend(items)
            total = results.get("total", 0)
            if len(all_items) >= total:
                break
            page += 1
        except Exception as e:
            if not all_items:
                raise
            print(f"⚠ Deal search stopped at page {page + 1}: {e}. Continuing with {len(all_items)} result(s).")
            break

    if not all_items:
        print("😕 No deals found right now. Try again later!")
        return []

    deals = []
    for item in all_items:
        try:
            price = float(item.get("price", {}).get("value", 0))
        except (TypeError, ValueError):
            continue
        if not (math.isfinite(price) and price > 0):
            continue

        discount_pct = ((market_stats["median"] - price) / market_stats["median"]) * 100
        z_score = ((price - market_stats["median"]) / market_stats["std_dev"]) if has_usable else None
        price_score = z_score if has_usable else -discount_pct
        gross_profit = market_stats["median"] - price
        fees = market_stats["median"] * EBAY_FEE_RATE
        net_profit = gross_profit - fees

        if has_usable:
            if z_score <= -2:
                signal = "🔥 Strong buy"
            elif z_score <= -1:
                signal = "✅ Buy"
            else:
                signal = "⚠️  Marginal"
        else:
            if discount_pct >= 20:
                signal = "🔥 Strong buy"
            elif discount_pct >= 10:
                signal = "✅ Buy"
            else:
                signal = "⚠️  Marginal"

        deals.append({
            "item_id": item.get("itemId", ""),
            "title": item.get("title", ""),
            "price": price,
            "z_score": z_score,
            "discount_pct": discount_pct,
            "price_score": price_score,
            "signal": signal,
            "gross_profit": gross_profit,
            "net_profit": net_profit,
            "link": item.get("itemWebUrl", ""),
        })

    deals.sort(key=lambda d: d["price_score"])
    print(f"📦 Found {len(deals)} deals across {page + 1} page(s)")
    return deals


deal_listings = find_deal_listings(SEARCH_QUERY, CONDITION, market_stats)


🔎 Searching active listings under $56.00 (z ≤ -1.0, i.e. 1.0σ below $75.00 median)...

📦 Found 52 deals across 1 page(s)


## Cell 9: Step 4 — Enrich Deals with Sold Quantity

In [10]:
def enrich_deals(deals):
    """Enrich each deal with sold quantity from Browse API getItem."""
    print(f"\n📦 Enriching {len(deals)} deals with sold quantity data...\n")

    for deal in deals:
        try:
            details = call_api_with_retry(
                f"Browse sold qty for {deal['item_id']}",
                lambda iid=deal["item_id"]: browse_get_item(iid),
            )
            avail = details.get("estimatedAvailabilities", [{}])
            sold_qty = avail[0].get("estimatedSoldQuantity", 0) if avail else 0
            deal["sold_quantity"] = sold_qty if isinstance(sold_qty, int) else 0
        except Exception as e:
            deal["sold_quantity"] = None
            print(f"⚠ Sold quantity enrichment failed for {deal['item_id']}: {e}")

    # Composite score: price score adjusted by sold volume
    for deal in deals:
        sold = deal["sold_quantity"] if isinstance(deal["sold_quantity"], int) else 0
        deal["composite_score"] = deal["price_score"] - 0.1 * sold

    deals.sort(key=lambda d: d["composite_score"])
    print(f"✅ Enrichment complete")
    return deals


enriched_deals = enrich_deals(deal_listings)


📦 Enriching 52 deals with sold quantity data...

✅ Enrichment complete


## Cell 10: Step 5 — LLM Evaluation (Optional)

In [11]:
def to_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def get_listing_details(item_id):
    """Get full listing details for LLM evaluation."""
    item = call_api_with_retry(
        f"Browse details for {item_id}",
        lambda: browse_get_item(item_id),
    )
    image_urls = [item.get("image", {}).get("imageUrl")]
    for img in to_list(item.get("additionalImages")):
        if isinstance(img, dict):
            image_urls.append(img.get("imageUrl"))
    image_urls = [u for u in image_urls if u]

    return {
        "title": item.get("title", ""),
        "description": item.get("description", "") or item.get("shortDescription", ""),
        "image_urls": image_urls,
        "condition_description": item.get("conditionDescription", ""),
        "condition_name": item.get("condition", ""),
        "item_specifics": [
            {"name": a.get("name", ""), "value": to_list(a.get("value"))}
            for a in to_list(item.get("localizedAspects"))
            if isinstance(a, dict)
        ],
        "seller": {
            "user_id": item.get("seller", {}).get("username", ""),
            "feedback_score": item.get("seller", {}).get("feedbackScore", 0),
            "positive_feedback_pct": item.get("seller", {}).get("feedbackPercentage", 0),
        },
    }


VALID_VERDICTS = {"BUY", "RISKY", "PASS"}

def parse_llm_response(content):
    """Parse and validate the LLM JSON response."""
    if not content or not content.strip():
        raise ValueError("LLM returned an empty response.")
    parsed = json.loads(content)
    verdict = str(parsed.get("verdict", "")).upper()
    confidence = float(parsed.get("confidence", -1))
    if verdict not in VALID_VERDICTS:
        raise ValueError(f"Invalid verdict: {parsed.get('verdict')}")
    if not (0 <= confidence <= 100):
        raise ValueError(f"Invalid confidence: {parsed.get('confidence')}")
    issues = parsed.get("issues", [])
    if not isinstance(issues, list) or any(not isinstance(i, str) for i in issues):
        raise ValueError("Invalid issues payload.")
    summary = str(parsed.get("summary", "")).strip()
    if not summary:
        raise ValueError("Invalid summary payload.")
    return {
        "verdict": verdict,
        "confidence": round(confidence),
        "issues": [i.strip() for i in issues if i.strip()],
        "summary": summary,
    }


def evaluate_listing_with_llm(client, listing_details, deal_context):
    """Evaluate a single listing with Azure OpenAI vision."""
    import re
    plain_desc = re.sub(r"<[^>]*>", " ", listing_details["description"])
    plain_desc = re.sub(r"\s+", " ", plain_desc).strip()[:3000]

    specs_text = "\n".join(
        f"{s['name']}: {', '.join(s['value']) if isinstance(s['value'], list) else s['value']}"
        for s in listing_details["item_specifics"]
    )

    z_ctx = (
        f"{deal_context['z_score']:.2f} (negative = below market)"
        if deal_context["z_score"] is not None
        else "N/A (market variance too low for z-score)"
    )
    disc_ctx = f"{deal_context['discount_pct']:.1f}% below median"

    content = [
        {
            "type": "text",
            "text": f"""You are an expert eBay arbitrage analyst specializing in evaluating "{SEARCH_QUERY}" listings for resale potential.

Your job is to determine whether this listing is actually a "{SEARCH_QUERY}" (not an accessory, case, cable, cover, or unrelated item), and if so, whether it's a good buy for resale.

LISTING DETAILS:
- Title: {listing_details['title']}
- Condition: {listing_details['condition_name']}
- Condition Notes: {listing_details['condition_description'] or 'None provided'}
- Seller: {listing_details['seller']['user_id']} ({listing_details['seller']['positive_feedback_pct']}% positive, {listing_details['seller']['feedback_score']} feedback score)
- Item Specifics:
{specs_text or 'None listed'}

DESCRIPTION:
{plain_desc or 'No description provided'}

DEAL CONTEXT:
- Listed price: ${deal_context['price']:.2f}
- Market median: ${deal_context['market_median']:.2f}
- Z-score: {z_ctx}
- Discount vs median: {disc_ctx}
- Estimated net profit: ${deal_context['net_profit']:.2f}

EVALUATE THIS LISTING FOR:
1. PRODUCT VERIFICATION: Is this actually a "{SEARCH_QUERY}"? If it's an accessory, case, cable, cover, replacement part, or different product entirely, verdict should be PASS regardless of price.
2. PHOTO ANALYSIS: Check all images for physical damage, screen issues, cracks, discoloration, missing parts.
3. DESCRIPTION ANALYSIS: Look for red-flag phrases ("as-is", "for parts", "not tested", "no returns"), functional issues.
4. SELLER ASSESSMENT: Grammar quality, disclosure transparency, feedback score.
5. COMPLETENESS: Are expected accessories included (charger, USB cable, slide cover, batteries)?
6. OVERALL VERDICT: Given the price vs. market value, is this a good buy for resale?""",
        },
    ]

    for url in listing_details["image_urls"][:MAX_IMAGES_PER_LISTING]:
        if url:
            content.append({"type": "image_url", "image_url": {"url": url}})

    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[{"role": "user", "content": content}],
        max_completion_tokens=800,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "listing_evaluation",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "verdict": {"type": "string", "enum": ["BUY", "RISKY", "PASS"]},
                        "confidence": {"type": "number", "minimum": 0, "maximum": 100},
                        "issues": {"type": "array", "items": {"type": "string"}},
                        "summary": {"type": "string"},
                    },
                    "required": ["verdict", "confidence", "issues", "summary"],
                    "additionalProperties": False,
                },
            },
        },
        timeout=LLM_TIMEOUT_S,
    )

    return parse_llm_response(response.choices[0].message.content)


def evaluate_deals_with_llm(deals, market_stats):
    """Evaluate top deals with Azure OpenAI vision (concurrent)."""
    if not LLM_ENABLED:
        print("\n💡 LLM evaluation skipped — set AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, and AZURE_OPENAI_DEPLOYMENT in .env to enable.\n")
        return deals

    client = AzureOpenAI(
        api_version="2024-10-01-preview",
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
    )

    to_evaluate = deals[:MAX_LLM_CALLS]
    if len(to_evaluate) < len(deals):
        print(f"  ℹ️  {len(deals)} deals found; evaluating top {len(to_evaluate)} (budget cap: {MAX_LLM_CALLS}).")
    print(f"\n🤖 Evaluating {len(to_evaluate)} deals with Azure OpenAI ({LLM_CONCURRENCY} parallel)...\n")

    completed = [0]

    def evaluate_one(deal, index):
        label = f"  [{index + 1}/{len(to_evaluate)}] {deal['title'][:40]}..."
        try:
            details = get_listing_details(deal["item_id"])
            evaluation = evaluate_listing_with_llm(client, details, {
                "price": deal["price"],
                "market_median": market_stats["median"],
                "z_score": deal["z_score"],
                "discount_pct": deal["discount_pct"],
                "net_profit": deal["net_profit"],
            })
            deal["llm_verdict"] = evaluation["verdict"]
            deal["llm_confidence"] = evaluation["confidence"]
            deal["llm_issues"] = evaluation["issues"]
            deal["llm_summary"] = evaluation["summary"]

            icon = {"BUY": "🟢", "RISKY": "🟡", "PASS": "🔴"}.get(evaluation["verdict"], "⚪")
            completed[0] += 1
            print(f"{label} {icon} {evaluation['verdict']} ({evaluation['confidence']}%) [{completed[0]}/{len(to_evaluate)}]")
        except Exception as e:
            deal["llm_verdict"] = "N/A"
            deal["llm_confidence"] = 0
            deal["llm_issues"] = []
            deal["llm_summary"] = f"Error: {e}"
            completed[0] += 1
            print(f"{label} ⚪ Error: {str(e)[:60]} [{completed[0]}/{len(to_evaluate)}]")

    with ThreadPoolExecutor(max_workers=LLM_CONCURRENCY) as executor:
        futures = [executor.submit(evaluate_one, deal, i) for i, deal in enumerate(to_evaluate)]
        for f in as_completed(futures):
            f.result()

    return deals


evaluated_deals = evaluate_deals_with_llm(enriched_deals, market_stats)


🤖 Evaluating 52 deals with Azure OpenAI (5 parallel)...

  [2/52] Texas Instruments TI-84 Plus CE Color Gr... 🟡 RISKY (1%) [1/52]
  [3/52] Texas Instruments TI-84 Plus CE Color Gr... 🟡 RISKY (1%) [2/52]
  [4/52] Texas Instruments TI-84 Plus CE Color Gr... 🔴 PASS (1%) [3/52]
  [1/52] Texas Instrument TI 84 Plus CE Ti Nspire... 🔴 PASS (1%) [4/52]
  [5/52] Texas Instrument TI 84 Plus CE Ti Nspire... 🔴 PASS (1%) [5/52]
  [6/52] TEXAS INSTRUMENTS TI-84 Plus CE Graphing... 🟡 RISKY (1%) [6/52]
  [7/52] Texas Instruments TI 84 Plus CE Ti Nspir... 🔴 PASS (1%) [7/52]
  [10/52] Texas Instruments TI-84 Plus CE Color Gr... 🟡 RISKY (1%) [8/52]
  [8/52] Texas Instruments TI-84 Plus CE Color Gr... 🟡 RISKY (1%) [9/52]
  [9/52] Texas Instruments TI-84 Plus CE Color Gr... 🟡 RISKY (1%) [10/52]
  [11/52] Texas Instruments TI-84 Plus CE Graphing... 🟡 RISKY (1%) [11/52]
  [15/52] Texas Instruments TI-84 Plus CE Color Gr... 🟡 RISKY (1%) [12/52]
  [12/52] Texas Instruments TI-84 Plus CE Color Gr... 🟡 RISKY (1

## Cell 11: Step 6 — Final Ranking & Display

In [12]:
def display_results(deals, market_stats):
    """Compute final recommendations and display results."""
    has_usable = market_stats["has_usable_std_dev"]
    has_llm = any(d.get("llm_verdict") and d["llm_verdict"] != "N/A" for d in deals)

    for deal in deals:
        price_rank = 3 if "Strong" in deal["signal"] else (2 if "Buy" in deal["signal"] else 1)
        llm_v = deal.get("llm_verdict", "")
        llm_rank = {"BUY": 3, "RISKY": 2, "PASS": 0}.get(llm_v, -1)
        confidence = deal.get("llm_confidence", 0)

        effective_llm = 0 if llm_rank == -1 else llm_rank
        deal["final_score"] = price_rank + effective_llm + (confidence / 100)

        if llm_rank == 0:
            deal["final_rec"] = "❌ SKIP"
        elif llm_rank == -1:
            if not has_llm:
                deal["final_rec"] = "✅ BUY" if price_rank >= 3 else ("⚠️  CONSIDER" if price_rank >= 2 else "❌ SKIP")
            else:
                deal["final_rec"] = "⚠️  CONSIDER" if price_rank >= 2 else "❌ SKIP"
        elif llm_rank == 3 and price_rank >= 2:
            deal["final_rec"] = "🏆 STRONG BUY"
        elif llm_rank == 3:
            deal["final_rec"] = "✅ BUY"
        elif llm_rank == 2 and price_rank >= 2:
            deal["final_rec"] = "⚠️  CONSIDER"
        else:
            deal["final_rec"] = "❌ SKIP"

    deals.sort(key=lambda d: d["final_score"], reverse=True)

    score_header = "Z-SCORE" if has_usable else "DISC%"
    print(f"\n📊 Full Analysis — {len(deals)} deals (best first):\n")

    rec_col = "RECOMMENDATION".ljust(18) if has_llm else ""
    header = (
        rec_col
        + score_header.ljust(10)
        + "SOLD".ljust(7)
        + ("VERDICT".ljust(14) if has_llm else "")
        + "SIGNAL".ljust(16)
        + "PRICE".ljust(10)
        + "NET PROFIT".ljust(12)
        + "TITLE".ljust(45)
        + "LINK"
    )
    print(header)
    print("─" * 150)

    for deal in deals:
        verdict_col = ""
        if has_llm:
            icon = {"BUY": "🟢", "RISKY": "🟡", "PASS": "🔴"}.get(deal.get("llm_verdict", ""), "⚪")
            verdict_col = f"{icon} {deal.get('llm_verdict', 'N/A')}".ljust(14)

        score_col = (
            f"{deal['z_score']:.2f}" if has_usable and deal["z_score"] is not None
            else f"{deal['discount_pct']:.1f}%"
        )
        sold_col = str(deal["sold_quantity"]) if isinstance(deal.get("sold_quantity"), int) else "?"
        rec_val = deal["final_rec"].ljust(18) if has_llm else ""
        profit_str = f"{'+'if deal['net_profit'] > 0 else ''}{format_currency(deal['net_profit'])}"

        print(
            rec_val
            + score_col.ljust(10)
            + sold_col.ljust(7)
            + verdict_col
            + deal["signal"].ljust(16)
            + format_currency(deal["price"]).ljust(10)
            + profit_str.ljust(12)
            + deal["title"][:42].ljust(45)
            + deal["link"]
        )

    print("─" * 150)

    # Legend
    print(f"\n💡 Column guide:")
    if has_usable:
        print("   Z-SCORE  = (price - median) / std dev — lower is cheaper")
    else:
        print("   DISC%    = ((median - price) / median) × 100")
    print("   SOLD     = units sold on this listing (demand signal)")
    if has_llm:
        print("   VERDICT  = LLM photo analysis: 🟢 BUY | 🟡 RISKY | 🔴 PASS")
        print("   RECOMMENDATION = combined price + LLM + confidence score")
    print(f"   NET PROFIT = (median - price) minus ~{EBAY_FEE_RATE * 100:.0f}% eBay fees (excl. shipping)\n")

    # Action items
    actionable = [d for d in deals if d["final_rec"] in ("🏆 STRONG BUY", "✅ BUY")]
    consider = [d for d in deals if d["final_rec"] == "⚠️  CONSIDER"]

    print("═" * 70)
    print("  🎯 ACTION ITEMS — Listings to buy NOW")
    print("═" * 70)

    if not actionable:
        print("\n  No strong recommendations right now.")
        print("  The market may be efficiently priced, or all cheap listings have issues.")
    else:
        for i, deal in enumerate(actionable):
            score_summary = (
                f"z={deal['z_score']:.2f}" if has_usable and deal["z_score"] is not None
                else f"{deal['discount_pct']:.1f}% off"
            )
            print(f"\n  {deal['final_rec']}  [{i + 1}]")
            print(f"  📦 {deal['title'][:70]}")
            print(f"  💰 {format_currency(deal['price'])} → est. profit {'+'if deal['net_profit'] > 0 else ''}{format_currency(deal['net_profit'])} ({score_summary})")
            if deal.get("llm_summary"):
                print(f"  🤖 {deal['llm_summary']}")
            print(f"  🔗 {deal['link']}")

    if consider:
        print("\n" + "─" * 70)
        print("  ⚠️  WORTH CONSIDERING (proceed with caution)")
        print("─" * 70)
        for i, deal in enumerate(consider):
            score_summary = (
                f"z={deal['z_score']:.2f}" if has_usable and deal["z_score"] is not None
                else f"{deal['discount_pct']:.1f}% off"
            )
            print(f"\n  [{i + 1}] {deal['title'][:65]}")
            print(f"      {format_currency(deal['price'])} | profit: {'+'if deal['net_profit'] > 0 else ''}{format_currency(deal['net_profit'])} ({score_summary})")
            if deal.get("llm_summary"):
                print(f"      🤖 {deal['llm_summary']}")
            if deal.get("llm_issues"):
                for issue in deal["llm_issues"]:
                    print(f"      ⚠ {issue}")
            print(f"      🔗 {deal['link']}")

    skipped = sum(1 for d in deals if d["final_rec"] == "❌ SKIP")
    print(f"\n  📊 Summary: {len(actionable)} BUY | {len(consider)} CONSIDER | {skipped} SKIP out of {len(deals)} deals")
    print("═" * 70 + "\n")


display_results(evaluated_deals, market_stats)


📊 Full Analysis — 52 deals (best first):

RECOMMENDATION    Z-SCORE   SOLD   VERDICT       SIGNAL          PRICE     NET PROFIT  TITLE                                        LINK
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
⚠️  CONSIDER      -3.61     0      🟡 RISKY       🔥 Strong buy    $7.00     +$58.25     Texas Instruments TI-84 Plus CE Color Grap   https://www.ebay.com/itm/205986161656?_skw=%22TI-84+Plus+CE%22+graphing+calculator+-charger+-case+-cable+-cover+-emulator+-software+-cord+-bag+-pouch+-dock+-screen+-protector+-%22parts+only%22+-%22for+parts%22+-stand+-holder+-skin+-sticker+-keypad&hash=item2ff5bb63f8:g:qroAAeSwSL1pYadt
⚠️  CONSIDER      -3.45     0      🟡 RISKY       🔥 Strong buy    $10.00    +$55.25     Texas Instruments TI-84 Plus CE Color Grap   https://www.ebay.com/itm/287251618308?_skw=%22TI-84+Plus+CE%22+graphing+calculator+-charger+-case+-cable+-cover+-emula